# φX174 812节点图：Edge-Cycle QUBO的SQA结果

**报告日期：2026-09-07**

> 节点数核对：工作区中没有821节点图或对应审计文件。当前完成图处理、Hamilton环验证并用于QUBO测试的是 **812节点、8,385条有向边** 的图，因此本报告按实际812节点数据撰写。

本报告聚焦最终数据输入、固定随机切口、Edge-Cycle QUBO定义、规模约化与OpenJij SQA结果。**随机初态SQA是主要结果**；已知证书初始化的SQA仅用于可行盆地诊断，经典tabu仅作为辅助基线。当前没有物理QA运行数据。

## 1. 数据与提取过程

输入为 `SRR27862880_phiX174_OLC_cycle742_pilot` 中的994条未纠错短reads；目录名中的 `cycle742` 来自原数据包的另一套string-graph处理，不代表本次QUBO输入节点数。

本次输入图的提取流程为：

1. 对994条reads执行minimap2 all-vs-all：`-k15 -w5 -m40 -n2 -X --secondary=yes -N1000 -c --eqx`。
2. 直接使用PAF几何建立候选边，不以DP结果拒绝边。
3. 使用 `identity ≥ 0.990`、`overlap ≥ 80 bp`。
4. 根据PAF相对方向约束，为每条物理read确定一个方向；方向约束无冲突。
5. 严格全覆盖/共线冗余筛选删除148条reads，保留846条。
6. 一次性删除34条缺少必要入边或出边支持的reads，不递归剥离。
7. 不进行传递约简、不重连边、不将节点度数预处理为1。

最终QUBO输入为 **812个物理read节点、8,385条有向候选边**；图是单个强连通分量，并已由图边独立确认存在覆盖全部812点的Hamilton环。

## 2. 固定随机切口与DAG投影

现有Edge-Cycle Hamiltonian要求read-read候选图为DAG。为测试该Hamiltonian，在已确认的图论Hamilton环上使用固定伪随机种子选择一个切口：

| 参数 | 数值 |
|---|---:|
| 切口随机种子 | `20260904` |
| 切口位置 | `122` |
| 被切环边 | `SRR27862880.107001_m1 → SRR27862880.9001_m2` |
| 路径起点 | `SRR27862880.9001_m2` |
| 路径终点 | `SRR27862880.107001_m1` |

将证书在切口后旋转得到线性rank，并仅保留满足 `rank(u) < rank(v)` 的候选边。该投影把8,385条边缩减为6,186条DAG边，移除2,199条逆rank或跨切口边，同时保留证书中的全部811条连续路径边。

> 该切口位置是随机的，但DAG rank来自已有Hamilton环证书；因此这里正式报告的是Edge-Cycle QUBO的编码与求解基准，不是未知环的端到端推断。

## 3. Edge-Cycle QUBO Hamiltonian

设read集合为 $V$，DAG候选边集合为 $E$。定义二元变量：

- $y_{uv}$：是否选择read边 $u\rightarrow v$；
- $s_v$：是否选择void边 $\mathrm{void}\rightarrow v$；
- $t_v$：是否选择void边 $v\rightarrow\mathrm{void}$。

总Hamiltonian为：

$$
H = H_{\mathrm{degree}} + H_{\mathrm{reward}} + H_{\mathrm{select}}.
$$

度约束项：

$$
\begin{aligned}
H_{\mathrm{degree}}=A\Bigg[
&\sum_{v\in V}\left(1-s_v-\sum_{u:(u,v)\in E}y_{uv}\right)^2\\
+&\sum_{v\in V}\left(1-t_v-\sum_{w:(v,w)\in E}y_{vw}\right)^2\\
+&\left(1-\sum_{v\in V}s_v\right)^2
+\left(1-\sum_{v\in V}t_v\right)^2
\Bigg].
\end{aligned}
$$

候选边奖励与可选线性选边项：

$$
H_{\mathrm{reward}}=-B\sum_{(u,v)\in E}\hat r_{uv}y_{uv},
\qquad
H_{\mathrm{select}}=\lambda\sum_{(u,v)\in E}y_{uv}.
$$

$\hat r_{uv}$ 是归一化的 `overlap_len_power2` 得分。本次正式结果使用 $\lambda=0$。

所有约束满足时，每个read恰好一入一出，void恰好连接一个起点和一个终点。由于read-read候选边是DAG，不可能形成独立read子环，因此所选结构必然是一条覆盖全部reads的路径，并通过void闭合成单个Hamilton环。

## 4. SQA主实验参数

| 类别 | 参数 | 数值 |
|---|---|---:|
| 图 | read节点数 | 812 |
| 图 | 原始候选边数 | 8,385 |
| 图 | DAG候选边数 | 6,186 |
| 得分 | `score_mode` | `overlap_len_power2` |
| 得分 | `normalize_rewards` | `True` |
| 得分 | 归一化reward范围 | 0.284444444444–1 |
| Hamiltonian | $A$ / `degree_penalty` | 100 |
| Hamiltonian | $B$ / `edge_reward_scale` | 10 |
| Hamiltonian | $\lambda$ / `edge_selection_penalty` | 0 |
| 求解器 | backend | OpenJij SQA |
| 求解器 | 初态 | 随机 |
| 求解器 | `num_reads` | 1 |
| 求解器 | `num_sweeps` | 1,000 |
| 求解器 | Trotter数 $m$ | 2 |
| 求解器 | $\beta$ | 2.0 |
| 求解器 | $\gamma$ | 1.0 |
| 求解器 | schedule | `quartic` |
| 求解器 | seed | 3 |

## 5. QUBO规模与精确约化

完整模型为每条DAG边设置一个变量，并为每个read增加两条void边变量：

$$|E|+2|V|=6186+2\times812=7810. $$

完整QUBO包含721,978个非零二次项。

本次切口已固定起点和终点，因此精确代入全部1,624个void变量：

- 仅 `void → path_start` 为1；
- 仅 `path_end → void` 为1；
- 其余void变量均为0。

约化后：

| 指标 | 完整QUBO | 固定端点后 |
|---|---:|---:|
| 变量数 | 7,810 | 6,186 |
| 二次项数 | 721,978 | 51,074 |

在SQA主实验参数（$B=10$）下，证书在约化前后的能量均为 `-6820.95066667`，说明变量代入精确保留了目标函数。

## 6. SQA主要结果

| 指标 | 结果 |
|---|---:|
| solver energy | `-4788.77866667` |
| 选中read边 | 817（可行目标为811） |
| read入度约束违例 | 6 |
| read出度约束违例 | 6 |
| 是否为有效Edge-Cycle | `False` |
| 可改善的单比特翻转数 | 0 |
| 中性能量的单比特翻转数 | 0 |
| 最小单比特能量增量 | `+5.67511111111` |
| 求解阶段耗时 | 3.334 s |

随机初态SQA在已测试配置中的最佳样本比目标多选6条边，形成6个入度冲突和6个出度冲突，尚未得到可行Hamilton环。所有冲突都是度数为2的盈余，没有度数为0的缺失。该样本不存在降能或等能的单比特翻转，说明SQA停在严格单比特局部极小；修复冲突需要多条边协同交换。

### 可行盆地诊断（仍为SQA）

把已知Hamilton路径证书编码为初态，并使用 `late-linear` 调度（$s:0.8\rightarrow1.0$），其余关键参数保持 $A=100$、$B=10$、$m=2$、$\beta=2$、$\gamma=1$、1,000 sweeps。SQA保持证书能量 `-6820.95066667`，选择811条边，入/出度违例均为0，并与旋转证书一致。

这项测试说明QUBO中存在稳定的可行低能盆地，但由于初态直接来自已知证书，它不是一次独立发现Hamilton环的求解结果。随机初态SQA的主要困难是进入该盆地。

### 辅助经典基线

作为模型可行性的辅助核对，D-Wave tabu经典求解器在 $A=100$、$B=20$、$\lambda=0$ 下得到811条边、入/出度零违例并匹配旋转证书，耗时5.367 s。该结果只作辅助，不替代SQA主结果，也不是物理QA结果。

In [ ]:
# 可选：读取SQA主结果、局部极小诊断、证书初始化诊断和辅助tabu基线。
from pathlib import Path
import csv

project = Path.cwd()
if not (project / 'debug').exists() and (project / 'olc_overlap_framework' / 'debug').exists():
    project = project / 'olc_overlap_framework'

base = project / 'debug/qubo/cyclic_hamiltonian'

def read_key_values(path):
    values = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if ': ' in line:
            key, value = line.split(': ', 1)
            values[key] = value
    return values

with (base / 'cycle742_fixed_endpoint_sqa_schedule_sweep.tsv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle, delimiter='\t'))
sqa_main = next(
    row for row in rows
    if row['schedule'] == 'quartic'
    and row['gamma'] == '1.0'
    and row['trotter'] == '2'
    and row['beta_final'] == '2.0'
)

local_minimum = read_key_values(base / 'cycle742_fixed_endpoint_sqa_local_minimum.txt')
certificate_sqa = read_key_values(base / 'cycle742_sqa_initial_state_diagnostic.txt')
tabu_auxiliary = read_key_values(base / 'cycle742_cut_dag_edge_cycle_audit.txt')

{
    'sqa_main': sqa_main,
    'sqa_local_minimum': {
        key: local_minimum.get(key) for key in (
            'selected_edges', 'in_conflict_nodes', 'out_conflict_nodes'
        )
    },
    'sqa_certificate_initialization': {
        key: certificate_sqa.get(key) for key in (
            'solver_energy', 'selected_edges',
            'read_in_constraint_violations',
            'read_out_constraint_violations',
            'matches_rotated_certificate',
        )
    },
    'auxiliary_classical_tabu': {
        key: tabu_auxiliary.get(key) for key in (
            'solver_energy', 'solver_valid_edge_cycle',
            'solver_selected_edge_count', 'solver_anneal_sec',
        )
    },
}

## 7. 结论

本次结果确认：

1. `EdgeCycleCoverDAGQUBOHamiltonian` 能编码812节点cut-DAG上的Hamilton路径，并通过void节点将其闭合为单环。
2. 固定切口端点后，QUBO可从7,810变量、721,978个二次项精确约化为6,186变量、51,074个二次项。
3. 作为正式主结果，随机初态OpenJij SQA的最佳样本为817条边、入/出度各6个违例，未独立得到可行Edge-Cycle，并停在严格单比特局部极小。
4. 证书初始化的晚启动SQA能保持811条边和零违例，证明可行低能盆地存在；它仅是诊断，不计作独立求解成功。
5. 经典tabu得到的零违例解只作为QUBO编码和约化可行性的辅助基线；当前没有物理QA结果。

结果边界：DAG rank与固定端点来自既有Hamilton环证书，因此该结果不代表从原始有环候选图自主推断切口和全局顺序。